<a href="https://colab.research.google.com/github/shims79757-lang/Elevance-Skills-Projects/blob/main/Comparative_App_Performance_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime
import re
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ----------------------------------------------------------------------
# 1. LOAD DATASET
# ----------------------------------------------------------------------
DATA_URL = "https://raw.githubusercontent.com/shims79757-lang/Elevance-Skills-Projects/main/googleplaystore.csv"
apps_df = pd.read_csv(DATA_URL)
data = apps_df.copy()

# ----------------------------------------------------------------------
# 2. DATA CLEANING
# ----------------------------------------------------------------------

# (a) Clean Installs: remove '+' and ',', cast to numeric
data['Installs'] = (
    data['Installs']
    .astype(str)
    .str.replace('+', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
data['Installs'] = pd.to_numeric(data['Installs'], errors='coerce')

# (b) Clean Reviews: remove commas, cast to numeric
data['Reviews'] = (
    data['Reviews'].astype(str).str.replace(',', '', regex=False).str.strip()
)
data['Reviews'] = pd.to_numeric(data['Reviews'], errors='coerce')


# (c) Clean Size: convert MB and kB to uniform float in MB
def convert_size(val):
  val = str(val).strip()
  if val.endswith('M'):
    try:
      return float(val[:-1])
    except ValueError:
      return np.nan
  elif val.endswith('k'):
    try:
      return float(val[:-1]) / 1024.0
    except ValueError:
      return np.nan
  return np.nan


data['Size_MB'] = data['Size'].apply(convert_size)

# (d) Clean Price: strip '$' and ',', convert to numeric (default 0.0 for free)
data['Price'] = (
    data['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
data['Price'] = pd.to_numeric(data['Price'], errors='coerce').fillna(0.0)

# (e) Clean Rating: convert to float
data['Rating'] = pd.to_numeric(data['Rating'], errors='coerce')


# (f) Clean Android Version: extract base numeric version
def clean_android_ver(val):
  if pd.isna(val) or 'varies' in str(val).lower():
    return np.nan
  parts = re.findall(r'\d+', str(val))
  if not parts:
    return np.nan
  major = int(parts[0])
  minor = int(parts[1]) if len(parts) > 1 else 0
  patch = int(parts[2]) if len(parts) > 2 else 0
  # Returns numeric version (e.g., 4.0.3 -> 4.03, 4.1 -> 4.10)
  return major + (minor * 0.1) + (patch * 0.01)


data['Android_Ver_Num'] = data['Android Ver'].apply(clean_android_ver)

# (g) Calculate preliminary columns:
# - Paid app revenue: Installs * Price
data['Revenue'] = data['Installs'] * data['Price']

# - Engagement rate (%): (Reviews / Installs) * 100
data['Engagement_Rate'] = (data['Reviews'] / data['Installs']) * 100

# ----------------------------------------------------------------------
# 3. APPLY FILTERING CRITERIA
# ----------------------------------------------------------------------
# Requirements:
# 1. Installs >= 10,000
# 2. Revenue > $10,000 for paid apps (free apps are retained)
# 3. Android version > 4.0
# 4. Size > 15 MB
# 5. Content Rating == "Everyone"
# 6. App names containing <= 30 characters
filtered_df = data[
    (data['Installs'] >= 10000)
    & (
        (data['Type'] == 'Free')
        | ((data['Type'] == 'Paid') & (data['Revenue'] > 10000))
    )
    & (data['Android_Ver_Num'] > 4.0)
    & (data['Size_MB'] > 15)
    & (data['Content Rating'] == 'Everyone')
    & (data['App'].astype(str).str.strip().str.len() <= 30)
].copy()

# Deduplicate identical app listings within the same category
filtered_df = filtered_df.drop_duplicates(
    subset=['App', 'Category']
).reset_index(drop=True)

# ----------------------------------------------------------------------
# 4. SELECT TOP 5 CATEGORIES BY TOTAL INSTALLS
# ----------------------------------------------------------------------
top_5_categories = (
    filtered_df.groupby('Category')['Installs']
    .sum()
    .nlargest(5)
    .index.tolist()
)

analysis_df = filtered_df[
    filtered_df['Category'].isin(top_5_categories)
].copy()


# ----------------------------------------------------------------------
# 5. AGGREGATE RAW METRICS FOR FREE VS. PAID
# ----------------------------------------------------------------------
def calculate_metrics(sub_df):
  """Computes the 6 summary metrics for a given subset."""
  if len(sub_df) == 0:
    return {
        'Average Installs': 0.0,
        'Weighted Rating': 0.0,
        'Total Reviews': 0.0,
        'Average Size': 0.0,
        'Revenue': 0.0,
        'Engagement Rate': 0.0,
    }

  avg_installs = sub_df['Installs'].mean()
  tot_reviews = sub_df['Reviews'].sum()
  tot_installs = sub_df['Installs'].sum()

  # Weighted rating using review counts: sum(Rating * Reviews) / sum(Reviews)
  valid_r = sub_df.dropna(subset=['Rating'])
  if len(valid_r) > 0 and valid_r['Reviews'].sum() > 0:
    weighted_rating = (valid_r['Rating'] * valid_r['Reviews']).sum() / valid_r[
        'Reviews'
    ].sum()
  else:
    weighted_rating = valid_r['Rating'].mean() if len(valid_r) > 0 else 0.0

  avg_size = sub_df['Size_MB'].mean()
  tot_revenue = sub_df['Revenue'].sum()
  engagement_rate = (
      (tot_reviews / tot_installs * 100) if tot_installs > 0 else 0.0
  )

  return {
      'Average Installs': avg_installs,
      'Weighted Rating': weighted_rating,
      'Total Reviews': tot_reviews,
      'Average Size': avg_size,
      'Revenue': tot_revenue,
      'Engagement Rate': engagement_rate,
  }


# Generate comparison slices: Overall + each of the Top 5 categories
slices = ['Overall'] + top_5_categories
records = []

for cat in slices:
  for app_type in ['Free', 'Paid']:
    if cat == 'Overall':
      subset = analysis_df[analysis_df['Type'] == app_type]
    else:
      subset = analysis_df[
          (analysis_df['Category'] == cat) & (analysis_df['Type'] == app_type)
      ]

    m = calculate_metrics(subset)
    m['Category'] = cat
    m['Type'] = app_type
    records.append(m)

comp_df = pd.DataFrame(records)

# ----------------------------------------------------------------------
# 6. PERCENTILE-BASED NORMALIZATION & COMPOSITE SCORING
# ----------------------------------------------------------------------
metrics = [
    'Average Installs',
    'Weighted Rating',
    'Total Reviews',
    'Average Size',
    'Revenue',
    'Engagement Rate',
]

# Percentile normalization (0-100 scale)
for m in metrics:
  comp_df[m + '_pct'] = comp_df[m].rank(pct=True, method='average') * 100

norm_metric_cols = [m + '_pct' for m in metrics]

# Composite Performance Score: average of all 6 percentile metrics
comp_df['Composite Score'] = comp_df[norm_metric_cols].mean(axis=1)

# Format raw strings for custom hover tooltips
for m in metrics:
  if m in ['Average Installs', 'Total Reviews']:
    comp_df[m + '_raw'] = comp_df[m].apply(lambda v: f'{v:,.0f}')
  elif m == 'Weighted Rating':
    comp_df[m + '_raw'] = comp_df[m].apply(lambda v: f'{v:.2f} ★')
  elif m == 'Average Size':
    comp_df[m + '_raw'] = comp_df[m].apply(lambda v: f'{v:.1f} MB')
  elif m == 'Revenue':
    comp_df[m + '_raw'] = comp_df[m].apply(lambda v: f'${v:,.2f}')
  elif m == 'Engagement Rate':
    comp_df[m + '_raw'] = comp_df[m].apply(lambda v: f'{v:.2f}%')

# ----------------------------------------------------------------------
# 7. BUILD RADAR CHART WITH DROPDOWN AND ANNOTATIONS
# ----------------------------------------------------------------------
# Closed loop for radar axes
theta_axes = metrics + [metrics[0]]

fig = go.Figure()
buttons = []
annotations_dict = {}

for idx, cat in enumerate(slices):
  free_row = comp_df[
      (comp_df['Category'] == cat) & (comp_df['Type'] == 'Free')
  ].iloc[0]
  paid_row = comp_df[
      (comp_df['Category'] == cat) & (comp_df['Type'] == 'Paid')
  ].iloc[0]

  r_free = [free_row[m + '_pct'] for m in metrics] + [
      free_row[metrics[0] + '_pct']
  ]
  r_paid = [paid_row[m + '_pct'] for m in metrics] + [
      paid_row[metrics[0] + '_pct']
  ]

  hover_free = [free_row[m + '_raw'] for m in metrics] + [
      free_row[metrics[0] + '_raw']
  ]
  hover_paid = [paid_row[m + '_raw'] for m in metrics] + [
      paid_row[metrics[0] + '_raw']
  ]

  # Default visible is the first slice ('Overall')
  is_visible = idx == 0

  # Free apps trace
  fig.add_trace(
      go.Scatterpolar(
          r=r_free,
          theta=theta_axes,
          fill='toself',
          name=f'{cat} - Free Apps',
          visible=is_visible,
          line=dict(color='#2563EB', width=2),
          fillcolor='rgba(37, 99, 235, 0.25)',
          customdata=hover_free,
          hovertemplate=(
              '<b>%{theta}</b><br>'
              'Percentile: %{r:.1f}%<br>'
              'Actual Value: %{customdata}'
              '<extra>Free Apps</extra>'
          ),
      )
  )

  # Paid apps trace
  fig.add_trace(
      go.Scatterpolar(
          r=r_paid,
          theta=theta_axes,
          fill='toself',
          name=f'{cat} - Paid Apps',
          visible=is_visible,
          line=dict(color='#DC2626', width=2),
          fillcolor='rgba(220, 38, 38, 0.25)',
          customdata=hover_paid,
          hovertemplate=(
              '<b>%{theta}</b><br>'
              'Percentile: %{r:.1f}%<br>'
              'Actual Value: %{customdata}'
              '<extra>Paid Apps</extra>'
          ),
      )
  )

  # Calculate winner for annotation
  score_f = free_row['Composite Score']
  score_p = paid_row['Composite Score']

  if score_f > score_p:
    better_text = (
        f'Better Performer: Free Apps (Score: {score_f:.1f} vs {score_p:.1f})'
    )
  elif score_p > score_f:
    better_text = (
        f'Better Performer: Paid Apps (Score: {score_p:.1f} vs {score_f:.1f})'
    )
  else:
    better_text = f'Performance Tie (Score: {score_f:.1f})'

  annot = [
      dict(
          text=(
              f'<b>{better_text}</b><br>'
              f"<span style='font-size:11px; color:#4B5563;'>"
              f'Free Score: {score_f:.1f} | Paid Score: {score_p:.1f}'
              '</span>'
          ),
          xref='paper',
          yref='paper',
          x=0.5,
          y=-0.22,
          showarrow=False,
          font=dict(size=13, color='#111827'),
          bgcolor='#F3F4F6',
          bordercolor='#D1D5DB',
          borderwidth=1,
          borderpad=8,
      )
  ]
  annotations_dict[cat] = annot

  # Build visibility toggle mask for this button
  vis_mask = [False] * (len(slices) * 2)
  vis_mask[idx * 2] = True
  vis_mask[idx * 2 + 1] = True

  buttons.append(
      dict(
          label=f'{cat}',
          method='update',
          args=[
              {'visible': vis_mask},
              {
                  'title.text': (
                      '<b>Free vs. Paid Apps: Radar Performance Analysis</b>'
                      f'<br><sup>Comparison Slice: {cat} (Percentile-Normalized'
                      ' Metrics)</sup>'
                  ),
                  'annotations': annot,
              },
          ],
      )
  )

# Add dropdown menu to layout
fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.0,
            xanchor='left',
            y=1.24,
            yanchor='top',
            bgcolor='#FFFFFF',
            bordercolor='#D1D5DB',
            font=dict(size=12, color='#111827'),
        )
    ],
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100],
            ticksuffix='%',
            showline=False,
            gridcolor='#E5E7EB',
        ),
        angularaxis=dict(gridcolor='#E5E7EB', linecolor='#9CA3AF'),
    ),
    title=dict(
        text=(
            '<b>Free vs. Paid Apps: Radar Performance Analysis</b>'
            '<br><sup>Comparison Slice: Overall (Percentile-Normalized'
            ' Metrics)</sup>'
        ),
        x=0.5,
        y=0.96,
        xanchor='center',
    ),
    annotations=annotations_dict['Overall'],
    template='plotly_white',
    height=750,
    margin=dict(l=80, r=80, t=140, b=120),
    legend=dict(orientation='h', y=1.12, x=0.5, xanchor='center'),
)

# ----------------------------------------------------------------------
# 8. TIME-GATED DISPLAY (1:00 PM - 2:00 PM IST)
# ----------------------------------------------------------------------
current_time = datetime.now(ZoneInfo('Asia/Kolkata'))
print('Current IST:', current_time.strftime('%d %B %Y, %I:%M:%S %p'))

if 13 <= current_time.hour < 14:
  fig.show()
else:
  print(
      'Visualization unavailable.\n'
      'This graph can only be viewed between 1:00 PM and 2:00 PM IST.'
  )